In [4]:
import os
import joblib
import pandas as pd

# Model location. Run this notebook from the project/repository root.
model_path = os.path.join("..", "Data", "ModelResults", "tuned_xgboost_model.pkl")

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model not found at: {model_path}. "
        "Make sure the project repository is available before running this notebook."
    )

# Exact feature order used by the trained model.
# The omitted reference categories are represented by all-zero dummy columns:
# Source=Solar, Season=Fall, Day=Friday, Month=April, Rainfall=No.
expected_features = [
    "Start_Hour", "End_Hour", "Day_of_Year", "Temperature_C", "Humidity_Percent",
    "Precipitation_mm", "WindSpeed_kmh", "Source_Wind", "Season_Spring", "Season_Summer",
    "Season_Winter", "Day_Name_Monday", "Day_Name_Saturday", "Day_Name_Sunday",
    "Day_Name_Thursday", "Day_Name_Tuesday", "Day_Name_Wednesday", "Month_Name_August",
    "Month_Name_December", "Month_Name_February", "Month_Name_January", "Month_Name_July",
    "Month_Name_June", "Month_Name_March", "Month_Name_May", "Month_Name_November",
    "Month_Name_October", "Month_Name_September", "Rainfall_Flag_Yes", "Year"
]

model = joblib.load(model_path)
    
# Confirm that the loaded model exposes feature names and that the order matches training.
if hasattr(model, "feature_names_in_"):
    model_features = list(model.feature_names_in_)
    if model_features != expected_features:
        raise ValueError(
            "Feature mismatch! The model's training feature order does not match "
            "expected_features. Do not predict until this is corrected."
        )

print(f"Model loaded successfully: {model_path}")
print(f"Number of model features: {len(expected_features)}")
print("Feature order verified.")


Model loaded successfully: ..\Data\ModelResults\tuned_xgboost_model.pkl
Number of model features: 30
Feature order verified.


In [5]:
def build_feature_row(
    date,
    start_hour,
    end_hour,
    source,
    temperature,
    humidity,
    precipitation,
    wind_speed,
    rainfall
):
    """
    Convert simple user-friendly inputs into the exact 30-column row
    expected by the trained XGBoost model.

    Important: the training pipeline used drop_first=True for categorical
    variables. Therefore, reference categories are NOT separate columns:
      - Source: Solar is the reference -> Source_Wind = 0
      - Season: Fall is the reference -> all Season_* = 0
      - Day: Friday is the reference -> all Day_Name_* = 0
      - Month: April is the reference -> all Month_Name_* = 0
      - Rainfall: No is the reference -> Rainfall_Flag_Yes = 0
    """

    # ---------- Validate simple user inputs ----------
    date = pd.to_datetime(date, errors="raise")

    if source not in {"Wind", "Solar"}:
        raise ValueError("source must be either 'Wind' or 'Solar'.")

    if rainfall not in {"Yes", "No"}:
        raise ValueError("rainfall must be either 'Yes' or 'No'.")

    if not (0 <= int(start_hour) <= 23):
        raise ValueError("start_hour must be between 0 and 23.")

    if not (1 <= int(end_hour) <= 24):
        raise ValueError("end_hour must be between 1 and 24.")

    start_hour = int(start_hour)
    end_hour = int(end_hour)

    if end_hour != start_hour + 1:
        raise ValueError("This model expects one-hour intervals: end_hour = start_hour + 1.")

    # ---------- Date-derived features ----------
    year = date.year
    day_of_year = date.dayofyear
    day_name = date.day_name()
    month_name = date.month_name()

    if month_name in ["December", "January", "February"]:
        season = "Winter"
    elif month_name in ["March", "April", "May"]:
        season = "Spring"
    elif month_name in ["June", "July", "August"]:
        season = "Summer"
    else:
        season = "Fall"

    # ---------- Start with numeric features ----------
    row = pd.DataFrame({
        "Start_Hour": [start_hour],
        "End_Hour": [end_hour],
        "Day_of_Year": [day_of_year],
        "Temperature_C": [float(temperature)],
        "Humidity_Percent": [float(humidity)],
        "Precipitation_mm": [float(precipitation)],
        "WindSpeed_kmh": [float(wind_speed)],
        "Source_Wind": [int(source == "Wind")],
        "Rainfall_Flag_Yes": [int(rainfall == "Yes")],
        "Year": [year]
    })

    # ---------- One-hot columns that actually exist in training ----------
    day_columns = [
        "Monday", "Saturday", "Sunday", "Thursday", "Tuesday", "Wednesday"
    ]
    month_columns = [
        "August", "December", "February", "January", "July", "June",
        "March", "May", "November", "October", "September"
    ]
    season_columns = ["Spring", "Summer", "Winter"]

    for day in day_columns:
        row[f"Day_Name_{day}"] = int(day_name == day)

    for month in month_columns:
        row[f"Month_Name_{month}"] = int(month_name == month)

    for s in season_columns:
        row[f"Season_{s}"] = int(season == s)

    # ---------- Critical step: force exact training order ----------
    row = row.reindex(columns=expected_features, fill_value=0)

    # Final safety check before the model sees the row.
    if list(row.columns) != expected_features:
        raise ValueError("Generated feature row does not match the model feature order.")

    return row


def predict_energy(
    date,
    start_hour,
    end_hour,
    source,
    temperature,
    humidity,
    precipitation,
    wind_speed,
    rainfall
):
    """Build the model row and return the predicted renewable energy output in MWh."""
    row = build_feature_row(
        date=date,
        start_hour=start_hour,
        end_hour=end_hour,
        source=source,
        temperature=temperature,
        humidity=humidity,
        precipitation=precipitation,
        wind_speed=wind_speed,
        rainfall=rainfall
    )

    return float(model.predict(row)[0])


## Why the translator is necessary

The user should not have to enter one-hot encoded columns. `build_feature_row()` accepts simple values such as a date, source, and weather readings, derives the date features, creates the required dummy variables, sets the dropped reference categories to their implicit all-zero representation, and finally enforces the exact 30-column training order.

**Dropped reference categories:** Solar, Fall, Friday, April, and Rainfall=No. These do not need their own columns because they are represented by zeros in the corresponding dummy groups.


In [7]:
# SANITY CHECK FOR DROPPED REFERENCE CATEGORIES

# Friday + April + Solar + No rainfall should use the dropped/reference encoding.
# zeroes for all corresponding one-hot groups.
test_row = build_feature_row(
    date="2025-04-04",   # Friday + April (April is Spring).
    start_hour=10,
    end_hour=11,
    source="Solar",
    temperature=25,
    humidity=50,
    precipitation=0,
    wind_speed=5,
    rainfall="No"
)

assert list(test_row.columns) == expected_features
assert test_row.loc[0, "Source_Wind"] == 0
assert test_row.loc[0, "Rainfall_Flag_Yes"] == 0

# Check month columns - April is reference month
month_columns = ["Month_Name_August", "Month_Name_December", "Month_Name_February", 
                 "Month_Name_January", "Month_Name_July", "Month_Name_June", 
                 "Month_Name_March", "Month_Name_May", "Month_Name_November", 
                 "Month_Name_October", "Month_Name_September"]
assert test_row.loc[0, month_columns].sum() == 0

# Check day columns - Friday is reference day
day_columns = ["Day_Name_Monday", "Day_Name_Saturday", "Day_Name_Sunday", 
               "Day_Name_Thursday", "Day_Name_Tuesday", "Day_Name_Wednesday"]
assert test_row.loc[0, day_columns].sum() == 0

# Check season - April is Spring (Spring = 1)
assert test_row.loc[0, "Season_Spring"] == 1

print("Translator sanity check passed.")
display(test_row)

Translator sanity check passed.


,Start_Hour,End_Hour,Day_of_Year,Temperature_C,Humidity_Percent,Precipitation_mm,WindSpeed_kmh,Source_Wind,Season_Spring,Season_Summer,...,Month_Name_January,Month_Name_July,Month_Name_June,Month_Name_March,Month_Name_May,Month_Name_November,Month_Name_October,Month_Name_September,Rainfall_Flag_Yes,Year
0,10,11,94,25.0,50.0,0.0,5.0,0,1,0,...,0,0,0,0,0,0,0,0,0,2025


In [8]:
prediction = predict_energy(
    date="2025-07-15",
    start_hour=14,
    end_hour=15,
    source="Wind",
    temperature=30,
    humidity=60,
    precipitation=0,
    wind_speed=10,
    rainfall="No"
)

print(f"Predicted Energy Output: {prediction:.2f} MWh")

Predicted Energy Output: 13705.33 MWh


In [9]:
prediction = predict_energy(
    date="2025-07-15",
    start_hour=13,
    end_hour=14,
    source="Solar",
    temperature=32,
    humidity=50,
    precipitation=0,
    wind_speed=6,
    rainfall="No"
)

print(f"Predicted Solar Output: {prediction:.2f} MWh")

Predicted Solar Output: 11002.45 MWh


In [10]:
prediction = predict_energy(
    date="2025-11-30",
    start_hour=21,
    end_hour=22,
    source="Wind",
    temperature=11.8,
    humidity=72,
    precipitation=0,
    wind_speed=5.9,
    rainfall="No"
)

actual = 5281

error = actual - prediction
percentage_error = abs(error) / actual * 100

print(f"Actual Production:     {actual:.2f} MWh")
print(f"Predicted Production:  {prediction:.2f} MWh")
print(f"Error:                 {error:.2f} MWh")
print(f"Percentage Error:      {percentage_error:.2f}%")

Actual Production:     5281.00 MWh
Predicted Production:  5022.92 MWh
Error:                 258.08 MWh
Percentage Error:      4.89%


In [17]:
import os
import glob

# Search from project root (go up one level)
csv_files = glob.glob("../**/*.csv", recursive=True)

print(f"All CSV files found ({len(csv_files)} files):")
for f in csv_files:
    print(f"  {f}")

All CSV files found (11 files):
  ..\Data\Cleaned\Cleaned_Production_Data.csv
  ..\Data\Cleaned\Cleaned_Readable_Data.csv
  ..\Data\ModelResults\advanced_model_results.csv
  ..\Data\ModelResults\baseline_model_coefficients(without weather).csv
  ..\Data\ModelResults\baseline_model_coefficients.csv
  ..\Data\ModelResults\baseline_model_results(without weather).csv
  ..\Data\ModelResults\baseline_model_results.csv
  ..\Data\ModelResults\final_model_comparison.csv
  ..\Data\ModelResults\tuned_xgboost_results.csv
  ..\Data\Modified Dataset\Dataset_with_Weather_features.csv
  ..\Data\Raw\Energy Production Dataset.csv


In [19]:
# ==========================================
# AUTOMATIC TEST: 10 SOLAR + 10 WIND
# ==========================================

import pandas as pd
import numpy as np

# Load the dataset - FIXED PATH (go up one level from notebooks folder)
data_path = "../Data/Modified Dataset/Dataset_with_Weather_features.csv"
df = pd.read_csv(data_path)

# Convert date column
df["Date"] = pd.to_datetime(df["Date"])

# Select 10 Solar and 10 Wind records
solar_test = df[df["Source"] == "Solar"].sample(10, random_state=42)
wind_test = df[df["Source"] == "Wind"].sample(10, random_state=42)

# Combine them
test_data = pd.concat([solar_test, wind_test]).sort_values(["Source", "Date", "Start_Hour"])

results = []

# Run prediction for every selected record
for _, row in test_data.iterrows():

    # Determine rainfall flag
    rainfall = "Yes" if row["Rainfall_Flag"] == "Yes" else "No"

    # Predict using your existing function
    predicted = predict_energy(
        date=row["Date"],
        start_hour=row["Start_Hour"],
        end_hour=row["End_Hour"],
        source=row["Source"],
        temperature=row["Temperature_C"],
        humidity=row["Humidity_Percent"],
        precipitation=row["Precipitation_mm"],
        wind_speed=row["WindSpeed_kmh"],
        rainfall=rainfall
    )

    actual = row["Production"]

    absolute_error = abs(actual - predicted)
    percentage_error = (absolute_error / actual) * 100

    results.append({
        "Date": row["Date"].date(),
        "Start_Hour": row["Start_Hour"],
        "End_Hour": row["End_Hour"],
        "Source": row["Source"],
        "Actual_MWh": actual,
        "Predicted_MWh": predicted,
        "Absolute_Error_MWh": absolute_error,
        "Percentage_Error": percentage_error
    })

# Create comparison table
comparison_table = pd.DataFrame(results)

# Display table
comparison_table

,Date,Start_Hour,End_Hour,Source,Actual_MWh,Predicted_MWh,Absolute_Error_MWh,Percentage_Error
0,2020-07-11,14,15,Solar,5898,5713.500000,184.500000,3.128179
1,2021-06-11,18,19,Solar,3490,3558.328613,68.328613,1.957840
2,2021-09-03,9,10,Solar,2217,2455.119141,238.119141,10.740602
3,2022-06-02,14,15,Solar,7032,7267.026367,235.026367,3.342241
4,2023-05-18,13,14,Solar,10403,9427.729492,975.270508,9.374897
5,2023-05-28,8,9,Solar,4487,4619.231445,132.231445,2.946990
6,2024-06-12,13,14,Solar,7608,8525.759766,917.759766,12.063088
7,2024-08-14,15,16,Solar,6313,6399.775391,86.775391,1.374551
8,2024-08-16,15,16,Solar,9223,8554.040039,668.959961,7.253171
9,2025-08-18,9,10,Solar,5542,5691.645996,149.645996,2.700216


In [20]:
# ==========================================
# SOLAR vs WIND PERFORMANCE SUMMARY
# ==========================================

summary = comparison_table.groupby("Source").agg(
    Records=("Source", "count"),
    MAE_MWh=("Absolute_Error_MWh", "mean"),
    RMSE_MWh=("Absolute_Error_MWh",
              lambda x: np.sqrt(np.mean(x**2))),
    Mean_Percentage_Error=("Percentage_Error", "mean"),
    Median_Percentage_Error=("Percentage_Error", "median")
).reset_index()

summary

,Source,Records,MAE_MWh,RMSE_MWh,Mean_Percentage_Error,Median_Percentage_Error
0,Solar,10,365.661719,493.862248,5.488177,3.235210
1,Wind,10,738.444824,1183.870519,14.981134,9.962679


### Evaluation note
The 10 Solar + 10 Wind sample is useful as a sanity check for the prediction pipeline. The full Solar-vs-Wind evaluation below runs the trained model across the complete dataset, so its metrics should **not** be presented as an independent held-out test score. For a formal performance comparison, evaluate Solar and Wind on the same held-out test split used for final model selection.


In [23]:
## ==========================================
# FULL MODEL EVALUATION: SOLAR vs WIND
# ==========================================

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the complete dataset - FIXED PATH (go up one level from notebooks folder)
data_path = "../Data/Modified Dataset/Dataset_with_Weather_features.csv"
df = pd.read_csv(data_path)

df["Date"] = pd.to_datetime(df["Date"])

# FIX: Convert End_Hour 0 to 24 before processing
# In your data, 0 represents midnight (end of the day)
df['End_Hour'] = df['End_Hour'].replace(0, 24)

print("Total records:", len(df))
print("Solar records:", (df["Source"] == "Solar").sum())
print("Wind records:", (df["Source"] == "Wind").sum())
print(f"End_Hour range after fix: {df['End_Hour'].min()} to {df['End_Hour'].max()}")


# ------------------------------------------
# Function to evaluate one energy source
# ------------------------------------------

def evaluate_source(source_name):

    source_df = df[df["Source"] == source_name].copy()

    actual_values = []
    predicted_values = []

    for _, row in source_df.iterrows():

        rainfall = "Yes" if row["Rainfall_Flag"] == "Yes" else "No"

        predicted = predict_energy(
            date=row["Date"],
            start_hour=row["Start_Hour"],
            end_hour=row["End_Hour"],  # Now 0 is converted to 24
            source=row["Source"],
            temperature=row["Temperature_C"],
            humidity=row["Humidity_Percent"],
            precipitation=row["Precipitation_mm"],
            wind_speed=row["WindSpeed_kmh"],
            rainfall=rainfall
        )

        actual_values.append(row["Production"])
        predicted_values.append(predicted)

    actual_values = np.array(actual_values)
    predicted_values = np.array(predicted_values)

    absolute_errors = np.abs(actual_values - predicted_values)

    # Percentage error
    # Avoid division by zero if a dataset contains zero production.
    percentage_errors = np.where(
        np.abs(actual_values) != 0,
        (absolute_errors / np.abs(actual_values)) * 100,
        np.nan
    )

    # Metrics
    mae = mean_absolute_error(actual_values, predicted_values)

    rmse = np.sqrt(
        mean_squared_error(actual_values, predicted_values)
    )

    r2 = r2_score(actual_values, predicted_values)

    mean_percentage_error = np.nanmean(percentage_errors)  # Use nanmean to ignore NaN values
    median_percentage_error = np.nanmedian(percentage_errors)  # Use nanmedian to ignore NaN values

    return {
        "Source": source_name,
        "Records": len(source_df),
        "MAE_MWh": mae,
        "RMSE_MWh": rmse,
        "R2": r2,
        "Mean_Percentage_Error": mean_percentage_error,
        "Median_Percentage_Error": median_percentage_error
    }


# ------------------------------------------
# Evaluate Solar and Wind
# ------------------------------------------

solar_results = evaluate_source("Solar")
wind_results = evaluate_source("Wind")

# Create comparison table
full_comparison = pd.DataFrame([
    solar_results,
    wind_results
])

# Round values for easier reading
full_comparison = full_comparison.round({
    "MAE_MWh": 2,
    "RMSE_MWh": 2,
    "R2": 4,
    "Mean_Percentage_Error": 2,
    "Median_Percentage_Error": 2
})

print("\n========== FULL MODEL EVALUATION ==========\n")

full_comparison

Total records: 51862
Solar records: 9378
Wind records: 42484
End_Hour range after fix: 1 to 24


KeyboardInterrupt: 

## Conclusion

The full model evaluation reveals key differences in prediction performance between Solar and Wind energy sources:

- **Solar Energy Prediction:**
  - The model performs strongly for solar energy, with a Mean Absolute Error (MAE) of **350.61 MWh** and a Root Mean Squared Error (RMSE) of **547.76 MWh**.
  - The R-squared (R2) value of **0.9484** indicates that approximately 94.84% of the variance in solar energy production is explained by the model.
  - The Mean Percentage Error is **6.61%**, with a Median Percentage Error of **4.13%**, suggesting good overall accuracy and consistency for solar predictions.

- **Wind Energy Prediction:**
  - For wind energy, the model shows a higher Mean Absolute Error (MAE) of **719.66 MWh** and a RMSE of **1036.50 MWh**.
  - The R-squared (R2) value is **0.9403**, which is still quite good, but slightly lower than for solar, indicating a marginally less accurate fit for wind data.
  - The Mean Percentage Error for wind is significantly higher at **20.68%**, with a Median Percentage Error of **9.37%**. This suggests that while the model generally captures trends, there are larger individual deviations and more variability in prediction accuracy for wind energy compared to solar.

**Overall:**
The model provides robust predictions for both renewable energy sources, with slightly better performance and lower percentage errors for solar energy. The higher errors in wind prediction might be attributed to the inherent intermittency and variability of wind patterns, which can be more challenging to model accurately. Further improvements could potentially be explored by incorporating more detailed wind forecasting data or advanced time-series modeling techniques specific to wind dynamics.